# 1. The acts of a year, from the dataset

[P_JOL_ROLL](https://github.com/Sarverott/P-jol-roll) keeps every act of
Dziennik Ustaw and Monitor Polski as tables in the dataset
[Sarverott/polish_national_law_docs](https://huggingface.co/datasets/Sarverott/polish_national_law_docs):
one line per act in `acts/<journal>/<year>.jsonl`.

The dataset is public: nothing here needs a token. Run it with
`uv run --project examples --with jupyterlab jupyter lab examples/notebooks` (or the marimo version: `task examples:convert`, then
version: `task examples:convert`, then
`uv run --project examples marimo edit examples/marimo`).

In [ ]:
import json

import pandas as pd
from huggingface_hub import hf_hub_download

DATASET = "Sarverott/polish_national_law_docs"


def table(path):
    """A file of the dataset, downloaded once and kept in the cache of huggingface_hub."""
    return hf_hub_download(DATASET, path, repo_type="dataset")

## The journals and their years

`lists/publishers.json` is the list the source gives: each journal, how many
acts it has and the years it was published in.

In [ ]:
with open(table("lists/publishers.json"), encoding="utf-8") as file:
    journals = pd.DataFrame(json.load(file))
journals["first"] = journals["years"].map(min)
journals["last"] = journals["years"].map(max)
journals[["code", "name", "actsCount", "first", "last"]]

## One year of Dziennik Ustaw

In [ ]:
JOURNAL, YEAR = "DU", 2025
acts = pd.read_json(table(f"acts/{JOURNAL}/{YEAR}.jsonl"), lines=True)
print(f"{len(acts)} acts in {JOURNAL} {YEAR}")
acts[["pos", "title", "type", "status", "promulgation"]].head(10)

## What kinds of acts, and are they still in force

In [ ]:
acts["type"].value_counts().head(10)

In [ ]:
acts["status"].value_counts()

## Acts by month of publication

In [ ]:
acts["month"] = pd.to_datetime(acts["promulgation"]).dt.to_period("M")
acts.groupby("month").size().plot.bar(title=f"Acts of {JOURNAL} {YEAR} by month")